In [ ]:
import psutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

RAM: 179.4 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
TV = torch.__version__.split('+')[0]
CU = torch.version.cuda.replace('.','') if torch.version.cuda else 'cpu'
print(f"PyTorch: {TV}, CUDA: {CU}")
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TV}+cu{CU}.html -q
!pip install torch-geometric pytorch-lightning==1.9.5 wandb jsonlines obonet pytorch-metric-learning -q
import os
os.environ['WANDB_MODE'] = 'disabled'
from torch_sparse import SparseTensor
print("✓ All packages installed")

PyTorch: 2.10.0, CUDA: 128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 141.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.2 MB/s eta 0:00:00
✓ All packages installed


In [ ]:
import os, sys
if os.path.exists('/content/SHEPHERD'):
    !rm -rf /content/SHEPHERD
!git clone https://github.com/mims-harvard/SHEPHERD.git /content/SHEPHERD
sys.path.insert(0, '/content/SHEPHERD')
sys.path.insert(0, '/content/SHEPHERD/shepherd')

SD = "/content/drive/MyDrive/Colab Notebooks/dataverse_shepherd"
PF = "/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients.jsonl"

with open('/content/SHEPHERD/project_config.py', 'w') as f:
    f.write(f'''from pathlib import Path
PROJECT_DIR = Path("{SD}")
CURR_KG = '8.9.21_kg'
KG_DIR = PROJECT_DIR / 'knowledge_graph' / CURR_KG
PREDICT_RESULTS_DIR = PROJECT_DIR / 'results'
SEED = 33
MY_DATA_DIR = PROJECT_DIR / 'patients'
MY_TRAIN_DATA = MY_DATA_DIR / 'simulated_patients' / f"disease_split_train_sim_patients_{{CURR_KG}}.txt"
MY_VAL_DATA = MY_DATA_DIR / 'simulated_patients' / f"disease_split_val_sim_patients_{{CURR_KG}}.txt"
CORRUPT_TRAIN_DATA = MY_DATA_DIR / 'simulated_patients' / f"disease_split_train_sim_patients_{{CURR_KG}}_phencorrupt.txt"
CORRUPT_VAL_DATA = MY_DATA_DIR / 'simulated_patients' / f"disease_split_val_sim_patients_{{CURR_KG}}_phencorrupt.txt"
MY_TEST_DATA = "{PF}"
MY_SPL_DATA = "{SD}/patients/test_patients_spl_matrix.npy"
MY_SPL_INDEX_DATA = "{SD}/patients/test_patients_spl_index_dict.pkl"
''')
print("✓ project_config.py")

Cloning into '/content/SHEPHERD'...
remote: Enumerating objects: 534, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 534 (delta 166), reused 153 (delta 153), pack-reused 344 (from 1)
Receiving objects: 100% (534/534), 565.76 KiB | 18.25 MiB/s, done.
Resolving deltas: 100% (325/325), done.
✓ project_config.py


In [ ]:
import os

ALLENNLP = '''
try:
    from allennlp.modules.attention import CosineAttention, BilinearAttention, AdditiveAttention, DotProductAttention
except ImportError:
    import torch
    import torch.nn as nn
    class BilinearAttention(nn.Module):
        def __init__(self, vector_dim, matrix_dim, **kwargs):
            super().__init__()
            self._weight_matrix = nn.Parameter(torch.zeros(vector_dim, matrix_dim))
            self._bias = nn.Parameter(torch.zeros(1))
            nn.init.xavier_uniform_(self._weight_matrix)
        def forward(self, vector, matrix):
            intermediate = torch.matmul(vector, self._weight_matrix).unsqueeze(1)
            return torch.sum(intermediate * matrix, dim=-1) + self._bias
    class AdditiveAttention(nn.Module):
        def __init__(self, vector_dim, matrix_dim, **kwargs):
            super().__init__()
            self._w_matrix = nn.Parameter(torch.zeros(vector_dim, vector_dim))
            self._u_matrix = nn.Parameter(torch.zeros(matrix_dim, vector_dim))
            self._v_vector = nn.Parameter(torch.zeros(vector_dim))
            nn.init.xavier_uniform_(self._w_matrix)
            nn.init.xavier_uniform_(self._u_matrix)
        def forward(self, vector, matrix):
            expanded = vector.unsqueeze(1).expand_as(matrix)
            intermediate = torch.tanh(expanded @ self._w_matrix + matrix @ self._u_matrix)
            return intermediate @ self._v_vector
    class CosineAttention(nn.Module):
        def forward(self, vector, matrix):
            return nn.functional.cosine_similarity(vector.unsqueeze(1), matrix, dim=-1)
    class DotProductAttention(nn.Module):
        def forward(self, vector, matrix):
            return torch.matmul(matrix, vector.unsqueeze(-1)).squeeze(-1)
'''

# allennlp patches
for fpath in ["/content/SHEPHERD/shepherd/task_heads/gp_aligner.py", "/content/SHEPHERD/shepherd/task_heads/patient_nca.py"]:
    with open(fpath, 'r') as f: c = f.read()
    c = c.replace("from allennlp.modules.attention import CosineAttention, BilinearAttention, AdditiveAttention, DotProductAttention", ALLENNLP)
    with open(fpath, 'w') as f: f.write(c)

fpath = "/content/SHEPHERD/shepherd/utils/train_utils.py"
with open(fpath, 'r') as f: c = f.read()
c = c.replace("from allennlp.modules.attention import CosineAttention, BilinearAttention, AdditiveAttention, DotProductAttention", ALLENNLP)
c = c.replace("from allennlp.modules.attention.attention import Attention", "try:\n    from allennlp.modules.attention.attention import Attention\nexcept ImportError:\n    import torch.nn as nn\n    class Attention(nn.Module):\n        pass")
c = c.replace("from allennlp.nn import Activation", "try:\n    from allennlp.nn import Activation\nexcept ImportError:\n    import torch.nn as nn\n    Activation = nn.ReLU")
with open(fpath, 'w') as f: f.write(c)

# predict.py patches
fpath = "/content/SHEPHERD/shepherd/predict.py"
with open(fpath, 'r') as f: c = f.read()
c = c.replace("trainer = pl.Trainer(gpus=hparams['n_gpus'])", "trainer = pl.Trainer(accelerator='cpu', devices=1)")
c = c.replace("hparams.update({'inference_batch_size': len(dataset)})", "hparams.update({'inference_batch_size': min(8, len(dataset))})")
with open(fpath, 'w') as f: f.write(c)

# hparams.py patches
fpath = "/content/SHEPHERD/shepherd/hparams.py"
with open(fpath, 'r') as f: c = f.read()
c = c.replace("'num_workers': 4", "'num_workers': 0")
c = c.replace("hparams.update({'add_similar_patients' : False})", "hparams.update({'add_similar_patients' : False})\n    hparams.update({'alpha': 1.0})")
with open(fpath, 'w') as f: f.write(c)

# samplers.py patches — EdgeIndex/Adj + augment_genes guards
fpath = "/content/SHEPHERD/shepherd/samplers.py"
with open(fpath, 'r') as f: c = f.read()
c = c.replace(
    "from torch_geometric.data.sampler import EdgeIndex, Adj",
    "from collections import namedtuple\nEdgeIndex = namedtuple('EdgeIndex', ['edge_index', 'e_id', 'size'])\nAdj = namedtuple('Adj', ['adj_t', 'e_id', 'size'])")
c = c.replace(
    "        if self.hparams['augment_genes']:\n            sim_gene_node_idx, gene_sims, gene_degs, unique_sim_genes = self.get_similar_genes(patient_ids, candidate_gene_node_idx)",
    "        if self.hparams['augment_genes'] and self.gene_similarity_dict is not None:\n            sim_gene_node_idx, gene_sims, gene_degs, unique_sim_genes = self.get_similar_genes(patient_ids, candidate_gene_node_idx)")
c = c.replace(
    "        if 'augment_genes' in self.hparams and self.hparams['augment_genes']:\n            sim_gene_node_idx = [g + 1 for g in sim_gene_node_idx]",
    "        if 'augment_genes' in self.hparams and self.hparams['augment_genes'] and sim_gene_node_idx is not None:\n            sim_gene_node_idx = [g + 1 for g in sim_gene_node_idx]")
c = c.replace(
    "        if 'augment_genes' in self.hparams and self.hparams['augment_genes']:\n            data['batch_cand_gene_degs'] = pad_sequence(gene_degs, batch_first=True, padding_value=0)\n            data['batch_sim_gene_nid'] = pad_sequence(sim_gene_node_idx, batch_first=True, padding_value=0)\n            data['batch_sim_gene_sims'] = pad_sequence(gene_sims, batch_first=True, padding_value=0)",
    "        if 'augment_genes' in self.hparams and self.hparams['augment_genes'] and gene_degs is not None:\n            data['batch_cand_gene_degs'] = pad_sequence(gene_degs, batch_first=True, padding_value=0)\n            data['batch_sim_gene_nid'] = pad_sequence(sim_gene_node_idx, batch_first=True, padding_value=0)\n            data['batch_sim_gene_sims'] = pad_sequence(gene_sims, batch_first=True, padding_value=0)")
with open(fpath, 'w') as f: f.write(c)

# Lightning LayerNorm patch
import pytorch_lightning.core.saving as saving
_original_load_state = saving._load_state
def _patched_load_state(cls, checkpoint, strict=True, **cls_kwargs_new):
    import torch
    if "state_dict" in checkpoint:
        sd = checkpoint["state_dict"]
        for key in list(sd.keys()):
            if 'layer_norms' in key and ('weight' in key or 'bias' in key):
                if sd[key].shape == torch.Size([1]):
                    if '.0.' in key: target = 2048
                    elif '.1.' in key: target = 1024
                    else: continue
                    sd[key] = sd[key].expand(target).clone()
    return _original_load_state(cls, checkpoint, strict=strict, **cls_kwargs_new)
saving._load_state = _patched_load_state

print("✓ All patches applied")

✓ All patches applied


In [ ]:
import torch, numpy as np, pickle, time
import pytorch_lightning as pl
import wandb
wandb.init(mode="disabled")

import project_config
from shepherd.dataset import PatientDataset
from shepherd.samplers import PatientNeighborSampler
import preprocess
from hparams import get_predict_hparams
from train import get_model
import argparse

args = argparse.Namespace(
    edgelist='KG_edgelist_mask.txt', node_map='KG_node_map.txt',
    patient_data='my_data', run_type='disease_characterization',
    saved_node_embeddings_path='checkpoints/pretrain.ckpt',
    best_ckpt='checkpoints/disease_characterization.ckpt')

hparams = get_predict_hparams(args)
hparams['augment_genes'] = False
pl.seed_everything(hparams['seed'])

print("Loading KG...")
all_data, edge_attr_dict, nodes = preprocess.preprocess_graph(args)
n_nodes = len(nodes["node_idx"].unique())
gene_phen_dis_node_idx = torch.LongTensor(nodes.loc[nodes['node_type'].isin(['gene/protein', 'effect/phenotype', 'disease']), 'node_idx'].values)

print("Loading SPL...")
spl = np.load(project_config.PROJECT_DIR / 'patients' / hparams['spl'])
spl_index_path = project_config.PROJECT_DIR / 'patients' / hparams['spl_index']
spl_indexing_dict = pickle.load(open(str(spl_index_path), "rb")) if spl_index_path.exists() else None

dataset = PatientDataset(project_config.PROJECT_DIR / 'patients' / hparams['test_data'], time=hparams['time'])
hparams.update({'inference_batch_size': min(8, len(dataset))})
nid_to_spl_dict = {nid: idx for idx, nid in enumerate(nodes[nodes["node_type"] == "gene/protein"]["node_idx"].tolist())}

print("Building dataloader...")
dataloader = PatientNeighborSampler('predict', all_data.edge_index, all_data.edge_index[:,all_data.test_mask],
    sizes=[-1,10,5], patient_dataset=dataset, batch_size=hparams['inference_batch_size'], sparse_sample=0,
    all_edge_attributes=all_data.edge_attr, n_nodes=n_nodes, relevant_node_idx=gene_phen_dis_node_idx,
    n_cand_diseases=hparams['test_n_cand_diseases'], use_diseases=hparams['use_diseases'],
    nid_to_spl_dict=nid_to_spl_dict, gp_spl=spl, spl_indexing_dict=spl_indexing_dict,
    shuffle=False, num_workers=0, hparams=hparams, pin_memory=False)
print("✓ Dataloader built")

print("Loading model...")
model = get_model(args, hparams, None, all_data, edge_attr_dict, n_nodes, load_from_checkpoint=True)
print("✓ Model loaded")

import psutil
print(f"RAM used: {psutil.virtual_memory().used / 1e9:.1f} GB / {psutil.virtual_memory().total / 1e9:.1f} GB")

print("Running inference...")
model.eval()
model = model.cpu()
t0 = time.time()
results = []
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        print(f"  Batch {i+1}... (RAM: {psutil.virtual_memory().used/1e9:.1f}/{psutil.virtual_memory().total/1e9:.1f} GB)")
        result = model.predict_step(batch, i)
        results.append(result)
print(f"✓ Inference done in {time.time()-t0:.0f}s, {len(results)} batches")

INFO:lightning_fabric.utilities.seed:Global seed set to 33


Predict hparams:  {'seed': 33, 'n_gpus': 0, 'num_workers': 0, 'profiler': 'simple', 'pin_memory': False, 'time': False, 'log_gpu_memory': False, 'debug': False, 'augment_genes': True, 'n_sim_genes': 3, 'aug_gene_w': 0.5, 'wandb_save_dir': PosixPath('/content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/wandb'), 'saved_checkpoint_path': PosixPath('/content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/checkpoints/pretrain.ckpt'), 'test_n_cand_diseases': -1, 'candidate_disease_type': 'all_kg_nodes', 'only_hard_distractors': False, 'patient_similarity_type': 'gene', 'n_similar_patients': 2, 'model_type': 'patient_NCA', 'loss': 'patient_disease_NCA', 'use_diseases': True, 'add_cand_diseases': True, 'add_similar_patients': False, 'wandb_project_name': 'disease-heterogeneity', 'alpha': 1.0, 'train_data': PosixPath('/content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/patients/simulated_patients/disease_split_train_sim_patients_8.9.21_kg.txt'), 'validation_data': PosixPath('/conte

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint was not an allowed global by default. Please use `torch.serialization.add_safe_globals([pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint])` or the `torch.serialization.safe_globals([pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint])` context manager to allowlist t

In [ ]:
import torch
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load
print("✓ Patched torch.load weights_only=False")

✓ Patched torch.load weights_only=False


In [ ]:
# Get one batch and check what keys it has
with torch.no_grad():
    for batch in dataloader:
        if isinstance(batch, dict):
            print("Keys:", list(batch.keys()))
        else:
            print("Type:", type(batch))
            print("Attrs:", [a for a in dir(batch) if not a.startswith('_')])
        break

Type: <class 'torch_geometric.data.data.Data'>
Attrs: ['apply', 'apply_', 'batch', 'clone', 'coalesce', 'concat', 'connected_components', 'contains_isolated_nodes', 'contains_self_loops', 'contiguous', 'coo', 'cpu', 'csc', 'csr', 'cuda', 'debug', 'detach', 'detach_', 'edge_attr', 'edge_attrs', 'edge_index', 'edge_stores', 'edge_subgraph', 'edge_weight', 'face', 'from_dict', 'generate_ids', 'get_all_edge_attrs', 'get_all_tensor_attrs', 'get_edge_index', 'get_tensor', 'get_tensor_size', 'has_isolated_nodes', 'has_self_loops', 'is_coalesced', 'is_cuda', 'is_directed', 'is_edge_attr', 'is_node_attr', 'is_sorted', 'is_sorted_by_time', 'is_undirected', 'keys', 'multi_get_tensor', 'node_attrs', 'node_offsets', 'node_stores', 'num_edge_features', 'num_edge_types', 'num_edges', 'num_faces', 'num_features', 'num_node_features', 'num_node_types', 'num_nodes', 'pin_memory', 'pos', 'put_edge_index', 'put_tensor', 'record_stream', 'remove_edge_index', 'remove_tensor', 'requires_grad_', 'share_memory

In [ ]:
with torch.no_grad():
    for batch in dataloader:
        print("Keys:", list(batch.keys()))
        break

Keys: ['disease_names', 'phenotype_names', 'corr_gene_names', 'batch_size', 'disease_one_hot_labels', 'batch_cand_disease_nid', 'cand_disease_names', 'batch_corr_gene_nid', 'batch_pheno_nid', 'n_id', 'patient_ids', 'batch_cand_gene_nid', 'batch_disease_nid', 'adjs', 'cand_gene_names']


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.forward(batch.n_id, batch.adjs)

        print(f"outputs: {outputs.shape}")
        print(f"n_id: {batch.n_id.shape}, max: {batch.n_id.max()}")
        print(f"batch_pheno_nid: {batch.batch_pheno_nid.shape}, max: {batch.batch_pheno_nid.max()}")
        print(f"batch_cand_disease_nid: {batch.batch_cand_disease_nid.shape}, max: {batch.batch_cand_disease_nid.max()}")

        # Check: are batch_pheno_nid values local indices (into n_id) or global?
        pheno_ids = batch.batch_pheno_nid.unique()
        n_ids = set(batch.n_id.tolist())
        in_nid = sum(1 for p in pheno_ids.tolist() if p in n_ids)
        print(f"Pheno IDs in n_id: {in_nid}/{len(pheno_ids)}")

        # Are they already local indices (0..len(outputs))?
        print(f"Max pheno_nid: {batch.batch_pheno_nid.max()}, outputs size: {outputs.shape[0]}")

        break

outputs: torch.Size([44392, 512])
n_id: torch.Size([102435]), max: 105219
batch_pheno_nid: torch.Size([8, 156]), max: 69830
batch_cand_disease_nid: torch.Size([8, 2725]), max: 76027
Pheno IDs in n_id: 395/403
Max pheno_nid: 69830, outputs size: 44392


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.forward(batch.n_id, batch.adjs)

        # Create full-size pad_outputs indexed by global node ID
        n_total = int(batch.n_id.max()) + 2  # +2 for padding at index 0
        full_outputs = torch.zeros(n_total, outputs.size(1))

        # n_id maps: local position in outputs → global node ID
        # But outputs only has embeddings for the LAST n nodes (target nodes from sampling)
        # n_id has ALL nodes involved in sampling (target + neighbors)
        # outputs has len = batch.adjs[-1].size[1] (the innermost target nodes)

        n_out = outputs.size(0)
        n_nid = batch.n_id.size(0)
        print(f"outputs: {n_out}, n_id: {n_nid}")

        # The first n_out entries in n_id correspond to the output embeddings
        global_ids = batch.n_id[:n_out]
        full_outputs[global_ids + 1] = outputs  # +1 for padding offset

        # Now test indexing
        pheno_emb = full_outputs[batch.batch_pheno_nid.view(-1) + 1]
        print(f"pheno_emb: {pheno_emb.shape}, NaN: {pheno_emb.isnan().any()}")
        print(f"Zero rows: {(pheno_emb.sum(dim=-1) == 0).sum()} / {pheno_emb.shape[0]}")

        break

outputs: 44392, n_id: 101731
pheno_emb: torch.Size([1248, 512]), NaN: False
Zero rows: 1078 / 1248


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        for i, adj in enumerate(batch.adjs):
            print(f"Adj {i}: edge_index {adj.edge_index.shape}, size {adj.size}")
        print(f"outputs would be: {batch.adjs[-1].size[1]} (target of last layer)")
        print(f"n_id: {batch.n_id.shape}")

        # The GNN works in reverse: adjs[0] is outermost, adjs[-1] is innermost
        # After forward, outputs has size = adjs[0].size[1] (the original target batch)
        print(f"adjs[0].size[1] = {batch.adjs[0].size[1]}")

        break

Adj 0: edge_index torch.Size([2, 384266]), size (102071, 92915)
Adj 1: edge_index torch.Size([2, 569752]), size (92915, 78025)
Adj 2: edge_index torch.Size([2, 6587621]), size (78025, 44392)
outputs would be: 44392 (target of last layer)
n_id: torch.Size([102071])
adjs[0].size[1] = 92915


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.forward(batch.n_id, batch.adjs)

        n_out = outputs.size(0)  # 44392
        target_global_ids = batch.n_id[:n_out]

        # Check: what fraction of phenotype nodes are in target_global_ids?
        target_set = set(target_global_ids.tolist())
        pheno_ids = batch.batch_pheno_nid[batch.batch_pheno_nid != 0].unique().tolist()
        found = sum(1 for p in pheno_ids if p in target_set)
        print(f"Phenotype nodes in GNN output: {found}/{len(pheno_ids)}")

        disease_ids = batch.batch_cand_disease_nid[batch.batch_cand_disease_nid != 0].unique().tolist()
        found_d = sum(1 for d in disease_ids if d in target_set)
        print(f"Disease nodes in GNN output: {found_d}/{len(disease_ids)}")

        break

Phenotype nodes in GNN output: 131/402
Disease nodes in GNN output: 21621/21793


In [ ]:
!sed -n '60,100p' /content/SHEPHERD/shepherd/patient_nca_model.py

        # get masks
        phenotype_mask = (batch.batch_pheno_nid != 0)
        if self.hparams.hparams['loss'] == 'patient_disease_NCA': disease_mask = (batch.batch_cand_disease_nid != 0)
        else: disease_mask = None

        # index into outputs using phenotype & disease batch node idx
        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)
        if self.hparams.hparams['loss'] == 'patient_disease_NCA': 
            batch_sz, max_n_dx = batch.batch_cand_disease_nid.shape
            disease_embeddings = torch.index_select(pad_outputs, 0, batch.batch_cand_disease_nid.view(-1)).view(batch_sz, max_n_dx, -1)
        else: disease_embeddings = None

        t2 = time.time()
        phenotype_embedding, disease_embeddings, phenotype_mask, disease_mask, attn_weights = self.patient_model.forward(phenotype_embeddings, disease_embeddings, phenotype_mask, 

In [ ]:
!sed -n '364,397p' /content/SHEPHERD/shepherd/node_embedder_model.py

    def predict(self, data):
        n_id = torch.arange(self.node_emb.weight.shape[0], device=self.device)

        x = self.node_emb(n_id)

        gat_attn = []
        for i in range(len(self.convs)):
            
            # Update node embeddings
            x, (edge_i, alpha) = self.convs[i](x, data.edge_index.to(self.device), return_attention_weights=True) #

            edge_i = edge_i.detach().cpu()
            alpha = alpha.detach().cpu()
            edge_i[0,:] = n_id[edge_i[0,:]]
            edge_i[1,:] = n_id[edge_i[1,:]]
            gat_attn.append((edge_i, alpha))
            
            # Normalize
            if i != self.n_layers - 1:
                if self.norm_method in ["batch", "layer"]:
                    x = self.norms[i](x)
                elif self.norm_method == "batch_layer":
                    x = self.layer_norms[i](x)
                x = F.leaky_relu(x)
                if self.norm_method == "batch_layer":
                    x = self.batch_norms[i

In [ ]:
!grep -n "node_model.predict\|node_model.forward" /content/SHEPHERD/shepherd/patient_nca_model.py

56:        outputs, gat_attn = self.node_model.forward(batch.n_id, batch.adjs)
323:        outputs, gat_attn = self.node_model.predict(self.all_data)


In [ ]:
!sed -n '315,345p' /content/SHEPHERD/shepherd/patient_nca_model.py

                'test/cand_disease_names': None

            })
        
        return batch_results

    
    def inference(self, batch, batch_idx):
        outputs, gat_attn = self.node_model.predict(self.all_data)

        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1), device=outputs.device), outputs]) 

        # get masks
        phenotype_mask = (batch.batch_pheno_nid != 0)
        if self.hparams.hparams['loss'] == 'patient_disease_NCA': disease_mask = (batch.batch_cand_disease_nid != 0)
        else: disease_mask = None

        # index into outputs using phenotype & disease batch node idx
        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)
        if self.hparams.hparams['loss'] == 'patient_disease_NCA': 
            batch_sz, max_n_dx = batch.batch_cand_disease_nid.shape
            disease_embeddings = torch.index_select(pad_outpu

In [ ]:
with torch.no_grad():
    print("Running full-graph GNN (this is what predict actually uses)...")
    import psutil
    print(f"RAM before: {psutil.virtual_memory().used/1e9:.1f}/{psutil.virtual_memory().total/1e9:.1f} GB")

    outputs, gat_attn = model.node_model.predict(model.all_data)

    print(f"RAM after: {psutil.virtual_memory().used/1e9:.1f}/{psutil.virtual_memory().total/1e9:.1f} GB")
    print(f"outputs: {outputs.shape}, NaN: {outputs.isnan().any()}, NaN count: {outputs.isnan().sum()}")
    print(f"Sample: {outputs[0,:5]}")

Running full-graph GNN (this is what predict actually uses)...
RAM before: 12.3/179.4 GB
RAM after: 12.5/179.4 GB
outputs: torch.Size([105220, 512]), NaN: False, NaN count: 0
Sample: tensor([ 0.8774, -0.4491, -0.0366, -1.0859,  2.0319])


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        result = model.predict_step(batch, 0)
        scores_df = result[0]
        print(f"Scores shape: {scores_df.shape}")
        print(f"NaN in similarities: {scores_df['similarities'].isna().sum()}/{len(scores_df)}")
        print(scores_df.head())
        break

run_folder /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/checkpoints/patient_NCA/05_13_22:08:00:32_lr_1e-05_val_simulated_pats.disease_split_val_sim_pats_kg_8.9.21_kg_losstype_pd_NCA/shepherd_test_patients
   patient_id                                     diseases  similarities
0       27219                        acrofacial dysostosis           NaN
1       27219  Klippel-Feil syndrome 1, autosomal dominant           NaN
2       27219             contact dermatitis due to nickel           NaN
3       27219                       Hughes-Stovin syndrome           NaN
4       27219                     stenosis of lacrimal sac           NaN
logging results to run dir:  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/checkpoints/patient_NCA/05_13_22:08:00:32_lr_1e-05_val_simulated_pats.disease_split_val_sim_pats_kg_8.9.21_kg_losstype_pd_NCA/shepherd_test_patients
   patient_id                       phenotypes  degrees  attention
0       27219    Intellectual disability,

In [ ]:
with torch.no_grad():
    for batch in dataloader:
        # Run the inference path that predict_step uses
        outputs, gat_attn = model.node_model.predict(model.all_data)
        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1)), outputs])

        phenotype_mask = (batch.batch_pheno_nid != 0)
        disease_mask = (batch.batch_cand_disease_nid != 0)

        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)
        print(f"1. phenotype_embeddings: NaN={phenotype_embeddings.isnan().any()}")

        batch_sz, max_n_dx = batch.batch_cand_disease_nid.shape
        disease_embeddings = torch.index_select(pad_outputs, 0, batch.batch_cand_disease_nid.view(-1)).view(batch_sz, max_n_dx, -1)
        print(f"2. disease_embeddings: NaN={disease_embeddings.isnan().any()}")

        # Patient model forward
        phen_emb, dis_emb, phen_mask, dis_mask, attn_w = model.patient_model.forward(phenotype_embeddings, disease_embeddings, phenotype_mask, disease_mask)
        print(f"3. phen_emb after patient_model.forward: NaN={phen_emb.isnan().any()}")
        print(f"4. dis_emb after patient_model.forward: NaN={dis_emb.isnan().any()}")

        # Calc loss (this produces softmax)
        labels = batch.disease_one_hot_labels
        loss, softmax, labels2, cand_idx, cand_emb = model.patient_model.calc_loss(
            batch, phen_emb, dis_emb, dis_mask, labels, use_candidate_list=False)
        print(f"5. softmax: NaN={softmax.isnan().any()}, shape={softmax.shape}")
        print(f"6. softmax non-NaN count: {(~softmax.isnan()).sum()}/{softmax.numel()}")

        break

1. phenotype_embeddings: NaN=False
2. disease_embeddings: NaN=False
3. phen_emb after patient_model.forward: NaN=True
4. dis_emb after patient_model.forward: NaN=False
5. softmax: NaN=True, shape=torch.Size([108, 21793])
6. softmax non-NaN count: 21793/2353644


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.predict(model.all_data)
        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1)), outputs])

        phenotype_mask = (batch.batch_pheno_nid != 0)
        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)

        # Reproduce what patient_model.forward does
        batched_attn = model.patient_model.attn_vector.repeat(batch_sz, 1)
        attn_scores = model.patient_model.attention(batched_attn, phenotype_embeddings, phenotype_mask)

        print(f"attn_scores shape: {attn_scores.shape}")
        print(f"attn_scores NaN: {attn_scores.isnan().any()}")
        print(f"attn_scores -inf count: {(attn_scores == float('-inf')).sum()}/{attn_scores.numel()}")
        print(f"attn_scores sample [0]: {attn_scores[0,:10]}")
        print(f"phenotype_mask sample [0]: {phenotype_mask[0,:10]}")
        print(f"phenotype_mask True count per patient: {phenotype_mask.sum(dim=1)[:5]}")

        break

attn_scores shape: torch.Size([108, 156])
attn_scores NaN: False
attn_scores -inf count: 14348/16848
attn_scores sample [0]: tensor([-0.6035, -0.5505, -0.1729, -0.4917, -0.9500, -0.3954, -0.5843, -0.5629,
        -0.9423,  0.1410])
phenotype_mask sample [0]: tensor([True, True, True, True, True, True, True, True, True, True])
phenotype_mask True count per patient: tensor([ 73,  31, 156,  48,  15])


In [ ]:
# Is the attention still our patched version that handles masks?
print(f"Attention type: {type(model.patient_model.attention)}")

# Check phenotype embeddings directly
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.forward(batch.n_id, batch.adjs)

        n_out = outputs.size(0)
        target_set = set(batch.n_id[:n_out].tolist())
        pheno_ids = batch.batch_pheno_nid[batch.batch_pheno_nid != 0].unique().tolist()
        found = sum(1 for p in pheno_ids if p in target_set)
        print(f"Phenotype nodes in GNN output: {found}/{len(pheno_ids)}")

        # Check the pad_outputs indexing
        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1)), outputs])
        pheno_emb = pad_outputs[batch.batch_pheno_nid.view(-1)]
        zero_rows = (pheno_emb.abs().sum(dim=-1) == 0).sum()
        print(f"Zero phenotype rows: {zero_rows}/{pheno_emb.shape[0]}")

        # Also check: what does inference() produce?
        node_emb, gat_attn, phen_emb, dis_emb, phen_mask, dis_mask, attn_w = model.inference(batch, 0)
        print(f"phen_emb NaN: {phen_emb.isnan().any()}")
        print(f"phen_emb sample: {phen_emb[0,:5]}")

        break

Attention type: <class '__main__.BilinearAttention'>
Phenotype nodes in GNN output: 528/1392


IndexError: index 62467 is out of bounds for dimension 0 with size 46373

In [ ]:
# Check: with batch_size = all 108 patients, would it fit in 179 GB RAM?
# The current run uses ~7 GB with batch_size=8
# batch_size=108 means ~13x more nodes in the computation graph
# Rough estimate: ~90 GB — should fit in 179 GB

hparams['inference_batch_size'] = len(dataset)  # all 108
print(f"Setting batch_size = {hparams['inference_batch_size']}")

# Rebuild dataloader
dataloader = PatientNeighborSampler('predict', all_data.edge_index, all_data.edge_index[:,all_data.test_mask],
    sizes=[-1,10,5], patient_dataset=dataset, batch_size=hparams['inference_batch_size'], sparse_sample=0,
    all_edge_attributes=all_data.edge_attr, n_nodes=n_nodes, relevant_node_idx=gene_phen_dis_node_idx,
    n_cand_diseases=hparams['test_n_cand_diseases'], use_diseases=hparams['use_diseases'],
    nid_to_spl_dict=nid_to_spl_dict, gp_spl=spl, spl_indexing_dict=spl_indexing_dict,
    shuffle=False, num_workers=0, hparams=hparams, pin_memory=False)
print("✓ Dataloader rebuilt with full batch")

import psutil, time
model.eval()
model = model.cpu()
t0 = time.time()
results = []
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        ram = psutil.virtual_memory()
        print(f"  Batch {i+1}... (RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB)")
        result = model.predict_step(batch, i)
        results.append(result)
elapsed = time.time() - t0
print(f"✓ Inference done in {elapsed:.0f}s, {len(results)} batches")

# Save immediately
import pandas as pd
scores_dfs = [r[0] for r in results]
scores_df = pd.concat(scores_dfs).reset_index(drop=True)
scores_df.to_csv('/content/drive/MyDrive/Colab Notebooks/split/shepherd_disease_char_scores.csv', index=False)
print(f"✓ Scores saved: {scores_df.shape}")
print(f"NaN in similarities: {scores_df['similarities'].isna().sum()}/{len(scores_df)}")
print(scores_df.head())

In [ ]:
import torch

# Load disease_characterization checkpoint and check attention params
ckpt = torch.load(str(project_config.PROJECT_DIR / 'checkpoints' / 'disease_characterization.ckpt'),
                  map_location='cpu', weights_only=False)

for key in ckpt['state_dict']:
    if 'attention' in key.lower() or 'attn' in key.lower():
        v = ckpt['state_dict'][key]
        print(f"{key}: shape={v.shape}, mean={v.mean():.4f}, std={v.std():.4f}")

# Compare with what's in the model now
print("\n--- Current model ---")
for name, param in model.patient_model.named_parameters():
    if 'attention' in name or 'attn' in name:
        print(f"{name}: shape={param.shape}, mean={param.mean():.4f}, std={param.std():.4f}")

patient_model.attn_vector: shape=torch.Size([1, 512]), mean=-0.0030, std=0.0619
patient_model.attention._weight_matrix: shape=torch.Size([512, 512]), mean=0.0001, std=0.0442
patient_model.attention._bias: shape=torch.Size([1]), mean=0.0000, std=nan

--- Current model ---
attn_vector: shape=torch.Size([1, 512]), mean=-0.0030, std=0.0619
attention._weight_matrix: shape=torch.Size([512, 512]), mean=0.0001, std=0.0442
attention._bias: shape=torch.Size([1]), mean=0.0000, std=nan


/tmp/ipykernel_2620/1502294403.py:10: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  print(f"{key}: shape={v.shape}, mean={v.mean():.4f}, std={v.std():.4f}")
/tmp/ipykernel_2620/1502294403.py:16: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  print(f"{name}: shape={param.shape}, mean={param.mean():.4f}, std={param.std():.4f}")


In [ ]:
# allennlp BilinearAttention.forward does:
# intermediate = vector @ weight_matrix  (not unsqueeze)
# then: scores = intermediate @ matrix.T + bias
# Our version does: matmul(vector, weight).unsqueeze(1) then sum over last dim

# Let's test both approaches on actual data
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.predict(model.all_data)
        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1)), outputs])

        phenotype_mask = (batch.batch_pheno_nid != 0)
        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)

        W = model.patient_model.attention._weight_matrix
        b = model.patient_model.attention._bias
        v = model.patient_model.attn_vector.repeat(batch_sz, 1)

        # Our approach
        inter_ours = torch.matmul(v, W).unsqueeze(1)  # [108, 1, 512]
        scores_ours = torch.sum(inter_ours * phenotype_embeddings, dim=-1) + b  # [108, 156]

        # allennlp's actual approach: bilinear(vector, W, matrix)
        # = vector @ W @ matrix^T
        inter_allen = v @ W  # [108, 512]
        scores_allen = torch.bmm(inter_allen.unsqueeze(1), phenotype_embeddings.transpose(1, 2)).squeeze(1) + b  # [108, 156]

        print(f"Our scores: {scores_ours[0,:5]}")
        print(f"Allen scores: {scores_allen[0,:5]}")
        print(f"Match: {torch.allclose(scores_ours, scores_allen)}")

        break

Our scores: tensor([-0.6035, -0.5505, -0.1729, -0.4917, -0.9500])
Allen scores: tensor([-0.6035, -0.5505, -0.1729, -0.4917, -0.9500])
Match: False


In [ ]:
with torch.no_grad():
    for batch in dataloader:
        outputs, gat_attn = model.node_model.predict(model.all_data)
        pad_outputs = torch.cat([torch.zeros(1, outputs.size(1)), outputs])

        phenotype_mask = (batch.batch_pheno_nid != 0)
        disease_mask = (batch.batch_cand_disease_nid != 0)
        batch_sz, max_n_phen = batch.batch_pheno_nid.shape
        phenotype_embeddings = torch.index_select(pad_outputs, 0, batch.batch_pheno_nid.view(-1)).view(batch_sz, max_n_phen, -1)
        batch_sz, max_n_dx = batch.batch_cand_disease_nid.shape
        disease_embeddings = torch.index_select(pad_outputs, 0, batch.batch_cand_disease_nid.view(-1)).view(batch_sz, max_n_dx, -1)

        # Run patient model forward
        phen_emb, dis_emb, pm, dm, aw = model.patient_model.forward(phenotype_embeddings, disease_embeddings, phenotype_mask, disease_mask)

        print(f"phen_emb: {phen_emb.shape}, NaN: {phen_emb.isnan().any()}")
        print(f"dis_emb: {dis_emb.shape}, NaN: {dis_emb.isnan().any()}")

        # Manually compute what NCA loss does
        from pytorch_metric_learning.distances import LpDistance
        dist_fn = LpDistance()
        mat = dist_fn(phen_emb.unsqueeze(1), dis_emb)
        print(f"\nDistance matrix: {mat.shape}")
        print(f"Distance range: {mat.min():.2f} to {mat.max():.2f}")
        print(f"Distance std per patient: {mat.std(dim=-1)[:5]}")

        # Check if distances vary enough for softmax to differentiate
        softmax = torch.nn.functional.softmax(-mat.squeeze(1), dim=1)
        print(f"\nSoftmax range: {softmax.min():.6f} to {softmax.max():.6f}")
        print(f"Max softmax per patient: {softmax.max(dim=1).values[:5]}")

        break

phen_emb: torch.Size([108, 512]), NaN: False
dis_emb: torch.Size([108, 202, 512]), NaN: False


ValueError: embeddings must be a 2D tensor of shape (batch_size, embedding_size)

In [ ]:
with torch.no_grad():
    # phen_emb and dis_emb from previous cell are still available
    # phen_emb: [108, 512], dis_emb: [108, 202, 512]

    # Compute L2 distance: patient embedding vs each disease embedding
    dists = torch.cdist(phen_emb.unsqueeze(1), dis_emb).squeeze(1)  # [108, 202]
    print(f"Distances: {dists.shape}")
    print(f"Range: {dists.min():.2f} to {dists.max():.2f}")
    print(f"Std per patient: {dists.std(dim=-1)[:5]}")

    # Softmax of negative distances
    softmax = torch.nn.functional.softmax(-dists, dim=1)
    print(f"\nSoftmax range: {softmax.min():.6f} to {softmax.max():.6f}")
    print(f"Max softmax per patient: {softmax.max(dim=1).values[:5]}")
    print(f"Ratio max/min per patient: {(softmax.max(dim=1).values / softmax.min(dim=1).values)[:5]}")

    # But wait — the actual NCA loss uses construct_batch_labels which
    # reshapes across ALL patients. Let me check what the actual loss sees
    labels = batch.disease_one_hot_labels
    loss, sm, lab, cidx, cemb = model.patient_model.calc_loss(
        batch, phen_emb, dis_emb, dm, labels, use_candidate_list=False)

    print(f"\nActual softmax from calc_loss: {sm.shape}")
    print(f"Actual softmax range: {sm.min():.8f} to {sm.max():.8f}")
    print(f"Actual softmax NaN: {sm.isnan().any()}")

Distances: torch.Size([108, 202])
Range: 3.04 to 908.71
Std per patient: tensor([ 6.6912,  9.8051,  2.5285,  2.4480, 12.7214])

Softmax range: 0.000000 to 0.893134
Max softmax per patient: tensor([0.0808, 0.0586, 0.1291, 0.0935, 0.1055])
Ratio max/min per patient: tensor([1.0690e+15, 7.2193e+22, 9.1674e+05, 8.0256e+09, 2.5455e+31])

Actual softmax from calc_loss: torch.Size([108, 21793])
Actual softmax range: 0.00001082 to 0.00018987
Actual softmax NaN: False


In [ ]:
import json, numpy as np, pandas as pd, pickle

patients = [json.loads(l) for l in open("/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients.jsonl")]

with open(project_config.KG_DIR / f'mondo_to_name_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    mondo_to_name = pickle.load(f)

scores_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/split/shepherd_disease_char_scores.csv')

# Rank WITHIN each patient (not globally)
ranks = []
for p in patients:
    pid = p['id']
    true_mondo = [str(d) for d in p['true_diseases']]
    true_names = set(mondo_to_name.get(d, d) for d in true_mondo if d in mondo_to_name)

    pdf = scores_df[scores_df['patient_id'] == pid].copy()
    pdf['rank'] = pdf['similarities'].rank(ascending=False, method='min')

    matched = pdf[pdf['diseases'].isin(true_names)]
    if len(matched) > 0:
        best_rank = int(matched['rank'].min())
        ranks.append(best_rank)
    else:
        ranks.append(len(pdf))

ranks = np.array(ranks)
n_diseases = scores_df.groupby('patient_id').size().values[0]
print(f"Diseases per patient: {n_diseases}")
print(f"\n--- SHEPHERD Disease Characterization (per-patient ranking) ---")
print(f"Patients evaluated: {len(ranks)}")
print(f"Mean rank: {ranks.mean():.1f} / {n_diseases}")
print(f"Median rank: {np.median(ranks):.1f}")
print(f"MRR: {(1.0/ranks).mean():.4f}")
for k in [1, 5, 10, 20, 50, 100, 500]:
    print(f"Recall@{k}: {(ranks <= k).mean():.4f}")

Diseases per patient: 21793

--- SHEPHERD Disease Characterization (per-patient ranking) ---
Patients evaluated: 108
Mean rank: 14122.9 / 21793
Median rank: 16659.0
MRR: 0.0005
Recall@1: 0.0000
Recall@5: 0.0000
Recall@10: 0.0000
Recall@20: 0.0000
Recall@50: 0.0093
Recall@100: 0.0093
Recall@500: 0.0278


In [ ]:
import torch

# Check ALL normalization params in the checkpoint vs model
ckpt = torch.load(str(project_config.PROJECT_DIR / 'checkpoints' / 'pretrain.ckpt'),
                  map_location='cpu', weights_only=False)

print("=== Checkpoint norm params ===")
for key in sorted(ckpt['state_dict']):
    if 'norm' in key:
        v = ckpt['state_dict'][key]
        print(f"  {key}: shape={v.shape}")

print("\n=== Model norm params ===")
for name, param in model.node_model.named_parameters():
    if 'norm' in name:
        print(f"  {name}: shape={param.shape}")

print("\n=== Model norm buffers ===")
for name, buf in model.node_model.named_buffers():
    if 'norm' in name:
        print(f"  {name}: shape={buf.shape}")

=== Checkpoint norm params ===
  batch_norms.0.module.bias: shape=torch.Size([2048])
  batch_norms.0.module.num_batches_tracked: shape=torch.Size([])
  batch_norms.0.module.running_mean: shape=torch.Size([2048])
  batch_norms.0.module.running_var: shape=torch.Size([2048])
  batch_norms.0.module.weight: shape=torch.Size([2048])
  batch_norms.1.module.bias: shape=torch.Size([1024])
  batch_norms.1.module.num_batches_tracked: shape=torch.Size([])
  batch_norms.1.module.running_mean: shape=torch.Size([1024])
  batch_norms.1.module.running_var: shape=torch.Size([1024])
  batch_norms.1.module.weight: shape=torch.Size([1024])
  layer_norms.0.bias: shape=torch.Size([1])
  layer_norms.0.weight: shape=torch.Size([1])
  layer_norms.1.bias: shape=torch.Size([1])
  layer_norms.1.weight: shape=torch.Size([1])

=== Model norm params ===
  batch_norms.0.module.weight: shape=torch.Size([2048])
  batch_norms.0.module.bias: shape=torch.Size([2048])
  batch_norms.1.module.weight: shape=torch.Size([1024])


In [ ]:
import torch
from torch_geometric.nn import LayerNorm

# New PyG LayerNorm with expanded weights
ln_new = LayerNorm(2048)
# Set to expanded checkpoint values
ln_new.weight.data.fill_(0.5948)
ln_new.bias.data.fill_(-0.3487)

# Simulate old behavior: just scale and shift (no normalization, or different normalization)
x = torch.randn(100, 2048)

# New LayerNorm output
out_new = ln_new(x)

# What if old LayerNorm was just: x * weight + bias (element-wise, no normalization)?
out_simple = x * 0.5948 + (-0.3487)

# Or: standard LayerNorm then scale+shift
out_standard = torch.nn.functional.layer_norm(x, [2048]) * 0.5948 + (-0.3487)

print(f"New PyG LayerNorm: mean={out_new.mean():.4f}, std={out_new.std():.4f}")
print(f"Simple scale+shift: mean={out_simple.mean():.4f}, std={out_simple.std():.4f}")
print(f"Standard LN + scale: mean={out_standard.mean():.4f}, std={out_standard.std():.4f}")
print(f"\nNew vs Simple match: {torch.allclose(out_new, out_simple, atol=0.01)}")
print(f"New vs Standard match: {torch.allclose(out_new, out_standard, atol=0.01)}")

New PyG LayerNorm: mean=-0.3487, std=0.5948
Simple scale+shift: mean=-0.3497, std=0.5946
Standard LN + scale: mean=-0.3487, std=0.5948

New vs Simple match: True
New vs Standard match: False


In [ ]:
# Compare GATv2Conv output on a small test to see if it's producing reasonable values
with torch.no_grad():
    # Get a few known disease nodes and their embeddings
    # Compare distances before and after GNN

    raw_emb = model.node_model.node_emb.weight  # [105220, 2048]
    gnn_out, _ = model.node_model.predict(model.all_data)  # [105220, 512]

    # Pick 5 disease nodes
    import pickle
    with open(project_config.KG_DIR / f'mondo_to_idx_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
        mondo_to_idx = pickle.load(f)

    # Get a test patient's true diseases and phenotypes
    p = patients[0]
    true_dis_idx = [mondo_to_idx[str(d)] for d in p['true_diseases'] if str(d) in mondo_to_idx][:3]
    pheno_idx = [hpo_to_idx[h] for h in p['positive_phenotypes'] if h in hpo_to_idx][:3]

    # Random disease for comparison
    import random
    all_dis = list(mondo_to_idx.values())
    random_dis_idx = random.sample(all_dis, 3)

    print("GNN embeddings - True diseases vs phenotypes:")
    for d in true_dis_idx:
        for ph in pheno_idx:
            dist = torch.dist(gnn_out[d], gnn_out[ph])
            print(f"  Disease {d} <-> Pheno {ph}: {dist:.2f}")

    print("\nGNN embeddings - Random diseases vs same phenotypes:")
    for d in random_dis_idx:
        for ph in pheno_idx:
            dist = torch.dist(gnn_out[d], gnn_out[ph])
            print(f"  Disease {d} <-> Pheno {ph}: {dist:.2f}")

NameError: name 'hpo_to_idx' is not defined

In [ ]:
import pickle
with open(project_config.KG_DIR / f'hpo_to_idx_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    hpo_to_idx = pickle.load(f)

# Now re-run
with torch.no_grad():
    raw_emb = model.node_model.node_emb.weight
    gnn_out, _ = model.node_model.predict(model.all_data)

    p = patients[0]
    true_dis_idx = [mondo_to_idx[str(d)] for d in p['true_diseases'] if str(d) in mondo_to_idx][:3]
    pheno_idx = [hpo_to_idx[h] for h in p['positive_phenotypes'] if h in hpo_to_idx][:3]

    import random
    all_dis = list(mondo_to_idx.values())
    random.seed(42)
    random_dis_idx = random.sample(all_dis, 3)

    print("GNN embeddings - True diseases vs phenotypes:")
    for d in true_dis_idx:
        for ph in pheno_idx:
            dist = torch.dist(gnn_out[d], gnn_out[ph])
            print(f"  Disease {d} <-> Pheno {ph}: {dist:.2f}")

    print("\nGNN embeddings - Random diseases vs same phenotypes:")
    for d in random_dis_idx:
        for ph in pheno_idx:
            dist = torch.dist(gnn_out[d], gnn_out[ph])
            print(f"  Disease {d} <-> Pheno {ph}: {dist:.2f}")

GNN embeddings - True diseases vs phenotypes:
  Disease 19751 <-> Pheno 60506: 19.83
  Disease 19751 <-> Pheno 18681: 19.56
  Disease 19751 <-> Pheno 60354: 10.36
  Disease 20154 <-> Pheno 60506: 15.08
  Disease 20154 <-> Pheno 18681: 15.19
  Disease 20154 <-> Pheno 60354: 7.47
  Disease 20248 <-> Pheno 60506: 23.49
  Disease 20248 <-> Pheno 18681: 23.33
  Disease 20248 <-> Pheno 60354: 14.83

GNN embeddings - Random diseases vs same phenotypes:
  Disease 75746 <-> Pheno 60506: 57.47
  Disease 75746 <-> Pheno 18681: 55.37
  Disease 75746 <-> Pheno 60354: 57.99
  Disease 23335 <-> Pheno 60506: 12.23
  Disease 23335 <-> Pheno 18681: 12.34
  Disease 23335 <-> Pheno 60354: 15.21
  Disease 20506 <-> Pheno 60506: 12.33
  Disease 20506 <-> Pheno 18681: 12.45
  Disease 20506 <-> Pheno 60354: 13.46


In [ ]:
import json, pickle

with open(project_config.KG_DIR / f'hpo_to_idx_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    hpo_to_idx = pickle.load(f)
with open(project_config.KG_DIR / f'mondo_to_idx_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    mondo_to_idx = pickle.load(f)
with open(project_config.KG_DIR / f'mondo_to_name_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    mondo_to_name = pickle.load(f)

patients = [json.loads(l) for l in open("/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients.jsonl")]

# Mapping rates
total_p, matched_p, total_d, matched_d = 0, 0, 0, 0
for p in patients:
    for hpo in p['positive_phenotypes']:
        total_p += 1
        if hpo in hpo_to_idx: matched_p += 1
    for d in p['true_diseases']:
        total_d += 1
        if str(d) in mondo_to_idx: matched_d += 1

print(f"Phenotypes: {matched_p}/{total_p} ({matched_p/total_p*100:.1f}%)")
print(f"Diseases: {matched_d}/{total_d} ({matched_d/total_d*100:.1f}%)")

# Check: what are the true disease NAMES for patient 0?
p0 = patients[0]
print(f"\nPatient {p0['id']} true diseases (MONDO IDs): {p0['true_diseases'][:5]}")
for d in p0['true_diseases'][:5]:
    name = mondo_to_name.get(str(d), "NOT FOUND")
    idx = mondo_to_idx.get(str(d), "NOT FOUND")
    print(f"  {d} -> '{name}' (idx: {idx})")

# Check: what does the scores CSV call these diseases?
scores_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/split/shepherd_disease_char_scores.csv')
p0_scores = scores_df[scores_df['patient_id'] == p0['id']]
true_names = [mondo_to_name.get(str(d), '') for d in p0['true_diseases'] if str(d) in mondo_to_name]
print(f"\nTrue disease names: {true_names[:5]}")
matched_in_scores = p0_scores[p0_scores['diseases'].isin(true_names)]
print(f"Found in scores: {len(matched_in_scores)}/{len(true_names)}")
if len(matched_in_scores) > 0:
    print(matched_in_scores[['diseases','similarities']].sort_values('similarities', ascending=False))

Phenotypes: 2500/2500 (100.0%)
Diseases: 273/276 (98.9%)

Patient 27219 true diseases (MONDO IDs): ['7794', '9239', '9223', '13912', '13961']
  7794 -> 'hypogonadotropic hypogonadism 7 with or without anosmia' (idx: 19751)
  9239 -> 'hypogonadotropic hypogonadism 24 without anosmia' (idx: 20154)
  9223 -> 'hypogonadotropic hypogonadism 23 with or without anosmia' (idx: 20248)
  13912 -> 'hypogonadotropic hypogonadism 10 with or without anosmia' (idx: 20966)
  13961 -> 'hypogonadotropic hypogonadism 16 with or without anosmia' (idx: 21167)

True disease names: ['hypogonadotropic hypogonadism 7 with or without anosmia', 'hypogonadotropic hypogonadism 24 without anosmia', 'hypogonadotropic hypogonadism 23 with or without anosmia', 'hypogonadotropic hypogonadism 10 with or without anosmia', 'hypogonadotropic hypogonadism 16 with or without anosmia']
Found in scores: 25/25
                                                diseases  similarities
6204   hypogonadotropic hypogonadism 23 with or 

In [ ]:
import numpy as np

ranks_all = []
for p in patients:
    pid = p['id']
    true_mondo = [str(d) for d in p['true_diseases']]
    true_names = set(mondo_to_name.get(d, d) for d in true_mondo if d in mondo_to_name)

    pdf = scores_df[scores_df['patient_id'] == pid].copy()
    pdf = pdf.sort_values('similarities', ascending=False).reset_index(drop=True)
    pdf['rank'] = range(1, len(pdf) + 1)

    matched = pdf[pdf['diseases'].isin(true_names)]
    if len(matched) > 0:
        best_rank = int(matched['rank'].min())
        all_ranks = matched['rank'].values.tolist()
        ranks_all.append({'pid': pid, 'best_rank': best_rank, 'n_true': len(true_names),
                         'n_matched': len(matched), 'all_ranks': sorted(all_ranks)[:5]})

# Show per-patient results
print("Per-patient best rank (lower = better):")
for r in sorted(ranks_all, key=lambda x: x['best_rank'])[:20]:
    print(f"  Patient {r['pid']}: best rank {r['best_rank']}, matched {r['n_matched']}/{r['n_true']}, top ranks: {r['all_ranks']}")

best_ranks = np.array([r['best_rank'] for r in ranks_all])
print(f"\n--- Summary ---")
print(f"Mean best rank: {best_ranks.mean():.1f} / 21793")
print(f"Median best rank: {np.median(best_ranks):.1f}")
print(f"MRR: {(1.0/best_ranks).mean():.4f}")
for k in [1, 5, 10, 50, 100, 500, 1000, 5000]:
    print(f"Recall@{k}: {(best_ranks <= k).mean():.4f}")

Per-patient best rank (lower = better):
  Patient 31630: best rank 42, matched 1/1, top ranks: [42]
  Patient 28262: best rank 120, matched 7/7, top ranks: [120, 557, 1350, 1787, 2075]
  Patient 29387: best rank 245, matched 7/7, top ranks: [245, 273, 443, 645, 2464]
  Patient 27249: best rank 1105, matched 6/6, top ranks: [1105, 1428, 1464, 1689, 3347]
  Patient 27219: best rank 1170, matched 25/25, top ranks: [1170, 2121, 2182, 2416, 2514]
  Patient 28426: best rank 1287, matched 6/6, top ranks: [1287, 1584, 1878, 4218, 4488]
  Patient 31714: best rank 1387, matched 1/1, top ranks: [1387]
  Patient 29646: best rank 1770, matched 4/4, top ranks: [1770, 11114, 12646, 13546]
  Patient 32905: best rank 1804, matched 1/1, top ranks: [1804]
  Patient 27684: best rank 1995, matched 3/3, top ranks: [1995, 2198, 9496]
  Patient 28456: best rank 2111, matched 1/1, top ranks: [2111]
  Patient 27292: best rank 2683, matched 12/12, top ranks: [2683, 3620, 3693, 3897, 4225]
  Patient 27421: best r

In [ ]:
import pandas as pd, pickle, json, numpy as np

# Load SHEPHERD's KG
edges = pd.read_csv(f"{SD}/knowledge_graph/8.9.21_kg/KG_edgelist_mask.txt", sep='\t')
nodes = pd.read_csv(f"{SD}/knowledge_graph/8.9.21_kg/KG_node_map.txt", sep='\t')

with open(f"{SD}/knowledge_graph/8.9.21_kg/mondo_to_idx_dict_8.9.21_kg.pkl", 'rb') as f:
    mondo_to_idx = pickle.load(f)
with open(f"{SD}/knowledge_graph/8.9.21_kg/hpo_to_idx_dict_8.9.21_kg.pkl", 'rb') as f:
    hpo_to_idx = pickle.load(f)
with open(f"{SD}/knowledge_graph/8.9.21_kg/ensembl_to_idx_dict_8.9.21_kg.pkl", 'rb') as f:
    ensembl_to_idx = pickle.load(f)

idx_to_hpo = {v: k for k, v in hpo_to_idx.items()}
idx_to_ensembl = {v: k for k, v in ensembl_to_idx.items()}
idx_to_mondo = {v: k for k, v in mondo_to_idx.items()}

# Build node_idx -> node_type mapping
node_type_map = dict(zip(nodes['node_idx'], nodes['node_type']))

# For each edge, get source/target types
# edges have x_idx, y_idx, full_relation, mask
print(f"Edge relations: {edges['full_relation'].unique()[:10]}")

# Find disease-phenotype and disease-gene edges
dis_pheno_edges = edges[edges['full_relation'].str.contains('disease_phenotype_positive')]
dis_gene_edges = edges[edges['full_relation'].str.contains('disease_protein')]
print(f"Disease-phenotype edges: {len(dis_pheno_edges)}")
print(f"Disease-gene edges: {len(dis_gene_edges)}")

# Load old patients to get disease list
old_patients = [json.loads(l) for l in open("/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients.jsonl")]

# Rebuild patients using SHEPHERD's KG
new_patients = []
for p in old_patients:
    disease_mondos = [str(d) for d in p['true_diseases']]
    disease_node_idxs = [mondo_to_idx[d] for d in disease_mondos if d in mondo_to_idx]

    # Get phenotypes from SHEPHERD's KG
    phenos = set()
    for d_idx in disease_node_idxs:
        # Find edges where this disease connects to phenotypes
        connected = dis_pheno_edges[(dis_pheno_edges['x_idx'] == d_idx) | (dis_pheno_edges['y_idx'] == d_idx)]
        for _, row in connected.iterrows():
            other = row['y_idx'] if row['x_idx'] == d_idx else row['x_idx']
            if other in idx_to_hpo:
                phenos.add(idx_to_hpo[other])

    # Get genes from SHEPHERD's KG
    genes = set()
    for d_idx in disease_node_idxs:
        connected = dis_gene_edges[(dis_gene_edges['x_idx'] == d_idx) | (dis_gene_edges['y_idx'] == d_idx)]
        for _, row in connected.iterrows():
            other = row['y_idx'] if row['x_idx'] == d_idx else row['x_idx']
            if other in idx_to_ensembl:
                genes.add(idx_to_ensembl[other])

    new_patients.append({
        "id": p['id'],
        "positive_phenotypes": list(phenos),
        "true_diseases": disease_mondos,
        "true_genes": list(genes),
        "all_candidate_genes": list(genes),
        "disease_name": p.get('disease_name', '')
    })

# Stats
n_with_pheno = sum(1 for p in new_patients if len(p['positive_phenotypes']) > 0)
n_with_genes = sum(1 for p in new_patients if len(p['all_candidate_genes']) > 0)
avg_pheno = np.mean([len(p['positive_phenotypes']) for p in new_patients])
avg_genes = np.mean([len(p['all_candidate_genes']) for p in new_patients])

print(f"\n--- Rebuilt patients ---")
print(f"Patients with phenotypes: {n_with_pheno}/{len(new_patients)}")
print(f"Patients with genes: {n_with_genes}/{len(new_patients)}")
print(f"Avg phenotypes: {avg_pheno:.1f}")
print(f"Avg candidate genes: {avg_genes:.1f}")
print(f"\nSample patient:")
print(json.dumps(new_patients[0], indent=2)[:500])

# Save
out_path = "/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v2.jsonl"
with open(out_path, 'w') as f:
    for p in new_patients:
        f.write(json.dumps(p) + '\n')
print(f"\n✓ Saved {len(new_patients)} patients to {out_path}")

Edge relations: ['gene/protein;protein_protein;gene/protein'
 'effect/phenotype;phenotype_protein;gene/protein'
 'effect/phenotype;phenotype_phenotype;effect/phenotype'
 'disease;disease_phenotype_negative;effect/phenotype'
 'disease;disease_phenotype_positive;effect/phenotype'
 'disease;disease_protein;gene/protein' 'disease;disease_disease;disease'
 'biological_process;bioprocess_bioprocess;biological_process'
 'molecular_function;molfunc_molfunc;molecular_function'
 'cellular_component;cellcomp_cellcomp;cellular_component']
Disease-phenotype edges: 409558
Disease-gene edges: 172598

--- Rebuilt patients ---
Patients with phenotypes: 86/108
Patients with genes: 78/108
Avg phenotypes: 25.7
Avg candidate genes: 24.8

Sample patient:
{
  "id": 27219,
  "positive_phenotypes": [
    "HP:0001250",
    "HP:0000812",
    "HP:0008734",
    "HP:0000062",
    "HP:0000054",
    "HP:0410004",
    "HP:0008187",
    "HP:0000122",
    "HP:0030260",
    "HP:0100783",
    "HP:0004367",
    "HP:0000135

In [ ]:
# Check which patients are missing data
import json
patients_v2 = [json.loads(l) for l in open("/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v2.jsonl")]
no_pheno = [p['id'] for p in patients_v2 if len(p['positive_phenotypes']) == 0]
no_genes = [p['id'] for p in patients_v2 if len(p['all_candidate_genes']) == 0]
print(f"No phenotypes: {len(no_pheno)} patients")
print(f"No genes: {len(no_genes)} patients")
print(f"Have both: {sum(1 for p in patients_v2 if len(p['positive_phenotypes']) > 0 and len(p['all_candidate_genes']) > 0)}/108")

# Filter to patients with both
valid = [p for p in patients_v2 if len(p['positive_phenotypes']) > 0 and len(p['all_candidate_genes']) > 0]
out_path = "/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v3.jsonl"
with open(out_path, 'w') as f:
    for p in valid:
        f.write(json.dumps(p) + '\n')
print(f"✓ Saved {len(valid)} valid patients to v3")

No phenotypes: 22 patients
No genes: 30 patients
Have both: 77/108
✓ Saved 77 valid patients to v3


In [ ]:
# Update project_config to point to v3
PF3 = "/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v3.jsonl"

with open('/content/SHEPHERD/project_config.py', 'r') as f:
    c = f.read()
c = c.replace(
    'MY_TEST_DATA = "/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients.jsonl"',
    f'MY_TEST_DATA = "{PF3}"')
with open('/content/SHEPHERD/project_config.py', 'w') as f:
    f.write(c)

# Reload config
import sys
for mod in list(sys.modules.keys()):
    if 'project_config' in mod:
        del sys.modules[mod]
import project_config
print(f"✓ Config updated: {project_config.MY_TEST_DATA}")

# Regenerate SPL for v3 patients
import numpy as np, pandas as pd, pickle, json, networkx as nx, time

edges = pd.read_csv(project_config.KG_DIR / 'KG_edgelist_mask.txt', sep='\t')
nodes = pd.read_csv(project_config.KG_DIR / 'KG_node_map.txt', sep='\t')

patients_v3 = [json.loads(l) for l in open(PF3)]
print(f"Patients: {len(patients_v3)}")

with open(project_config.KG_DIR / f'hpo_to_idx_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    hpo_to_idx = pickle.load(f)

gene_nodes = np.sort(nodes[nodes['node_type'] == 'gene/protein']['node_idx'].values)
gene_to_col = {int(g): i for i, g in enumerate(gene_nodes)}

all_pheno_idx = set()
for p in patients_v3:
    for hpo in p['positive_phenotypes']:
        if hpo in hpo_to_idx:
            all_pheno_idx.add(hpo_to_idx[hpo])
print(f"Unique phenotypes: {len(all_pheno_idx)}")

G = nx.Graph()
G.add_edges_from(zip(edges['x_idx'].values.astype(int), edges['y_idx'].values.astype(int)))

pheno_idx_list = sorted(all_pheno_idx)
pheno_to_row = {p: i for i, p in enumerate(pheno_idx_list)}

t0 = time.time()
spl_rows = []
for i, pidx in enumerate(pheno_idx_list):
    if (i + 1) % 200 == 0:
        elapsed = time.time() - t0
        remaining = elapsed / (i + 1) * (len(pheno_idx_list) - i - 1)
        print(f"  {i+1}/{len(pheno_idx_list)} ({remaining:.0f}s remaining)")
    try:
        lengths = nx.single_source_shortest_path_length(G, int(pidx))
        row = np.full(len(gene_nodes), 99, dtype=np.float32)
        for gidx, col in gene_to_col.items():
            if gidx in lengths:
                row[col] = lengths[gidx]
        spl_rows.append(row)
    except:
        spl_rows.append(np.full(len(gene_nodes), 99, dtype=np.float32))

spl_matrix = np.stack(spl_rows)
print(f"SPL: {spl_matrix.shape} in {time.time()-t0:.0f}s")

import os
os.makedirs(f"{project_config.PROJECT_DIR}/patients", exist_ok=True)
np.save(str(project_config.PROJECT_DIR / 'patients' / 'test_patients_spl_matrix.npy'), spl_matrix)
with open(str(project_config.PROJECT_DIR / 'patients' / 'test_patients_spl_index_dict.pkl'), 'wb') as f:
    pickle.dump(pheno_to_row, f)
print("✓ SPL saved")

# Reload dataset and dataloader
for mod in list(sys.modules.keys()):
    if 'shepherd' in mod and 'dataset' in mod:
        del sys.modules[mod]
from shepherd.dataset import PatientDataset

dataset = PatientDataset(project_config.PROJECT_DIR / 'patients' / project_config.MY_TEST_DATA, time=False)
hparams['inference_batch_size'] = len(dataset)
hparams['augment_genes'] = False
nid_to_spl_dict = {nid: idx for idx, nid in enumerate(nodes[nodes["node_type"] == "gene/protein"]["node_idx"].tolist())}

spl = np.load(str(project_config.PROJECT_DIR / 'patients' / 'test_patients_spl_matrix.npy'))
spl_index_path = project_config.PROJECT_DIR / 'patients' / 'test_patients_spl_index_dict.pkl'
spl_indexing_dict = pickle.load(open(str(spl_index_path), "rb"))

dataloader = PatientNeighborSampler('predict', all_data.edge_index, all_data.edge_index[:,all_data.test_mask],
    sizes=[-1,10,5], patient_dataset=dataset, batch_size=hparams['inference_batch_size'], sparse_sample=0,
    all_edge_attributes=all_data.edge_attr, n_nodes=n_nodes, relevant_node_idx=gene_phen_dis_node_idx,
    n_cand_diseases=hparams['test_n_cand_diseases'], use_diseases=hparams['use_diseases'],
    nid_to_spl_dict=nid_to_spl_dict, gp_spl=spl, spl_indexing_dict=spl_indexing_dict,
    shuffle=False, num_workers=0, hparams=hparams, pin_memory=False)
print("✓ Dataloader rebuilt")

# Run inference
import psutil
model.eval()
model = model.cpu()
t0 = time.time()
results = []
with torch.no_grad():
    for i, batch in enumerate(dataloader):
        print(f"  Batch {i+1}... (RAM: {psutil.virtual_memory().used/1e9:.1f}/{psutil.virtual_memory().total/1e9:.1f} GB)")
        result = model.predict_step(batch, i)
        results.append(result)
elapsed = time.time() - t0
print(f"✓ Inference done in {elapsed:.0f}s")

# Save & evaluate
scores_df = result[0]
scores_df.to_csv('/content/drive/MyDrive/Colab Notebooks/split/shepherd_disease_char_scores_v3.csv', index=False)

nan_count = scores_df['similarities'].isna().sum()
print(f"Scores: {scores_df.shape}, NaN: {nan_count}/{len(scores_df)}")

# Evaluate
with open(project_config.KG_DIR / f'mondo_to_name_dict_{project_config.CURR_KG}.pkl', 'rb') as f:
    mondo_to_name = pickle.load(f)

ranks = []
for p in patients_v3:
    pid = p['id']
    true_names = set(mondo_to_name.get(str(d), str(d)) for d in p['true_diseases'] if str(d) in mondo_to_name)
    pdf = scores_df[scores_df['patient_id'] == pid].copy()
    pdf = pdf.sort_values('similarities', ascending=False).reset_index(drop=True)
    pdf['rank'] = range(1, len(pdf) + 1)
    matched = pdf[pdf['diseases'].isin(true_names)]
    if len(matched) > 0:
        ranks.append(int(matched['rank'].min()))
    else:
        ranks.append(len(pdf))

ranks = np.array(ranks)
print(f"\n--- SHEPHERD v3 (with genes from KG) ---")
print(f"Patients: {len(ranks)}")
print(f"Mean rank: {ranks.mean():.1f}")
print(f"Median rank: {np.median(ranks):.1f}")
print(f"MRR: {(1.0/ranks).mean():.4f}")
for k in [1, 5, 10, 50, 100, 500, 1000]:
    print(f"Recall@{k}: {(ranks <= k).mean():.4f}")

✓ Config updated: /content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v3.jsonl
Patients: 77
Unique phenotypes: 1433
  200/1433 (869s remaining)
  400/1433 (716s remaining)
  600/1433 (573s remaining)
  800/1433 (434s remaining)
  1000/1433 (296s remaining)
  1200/1433 (159s remaining)
  1400/1433 (22s remaining)
SPL: (1433, 21610) in 975s
✓ SPL saved
Dataset filepath:  /content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v3.jsonl
Number of patients:  77
Finished initalizing dataset
✓ Dataloader rebuilt
  Batch 1... (RAM: 16.2/179.4 GB)
run_folder /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/checkpoints/patient_NCA/05_13_22:08:00:32_lr_1e-05_val_simulated_pats.disease_split_val_sim_pats_kg_8.9.21_kg_losstype_pd_NCA/shepherd_test_patients
   patient_id                                           diseases  similarities
0       27219  dehydrated hereditary stomatocytosis with or w...      0.000068
1       27219         autosomal dominant familial p

In [ ]:
import pickle, json, numpy as np, pandas as pd

# Load Orphanet disease category mapping
# SHEPHERD categorizes diseases into 33 Orphanet categories
# We need the orphanet_to_mondo mapping and disease categories

# Check if we have the category mapping
import os
for root, dirs, files in os.walk(f"{SD}/preprocess"):
    for f in files:
        print(f"  {os.path.join(root, f)}")

  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/hp_terms.csv
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/.DS_Store
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/orphanet/categorization_of_orphanet_diseases.csv
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/orphanet/orphanet_final_disease_metadata.tsv
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/orphanet/orphanet_to_mondo_dict.pkl
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/orphanet/orphanet_to_omim_mapping_df.csv
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/mondo/mondo2hpo.csv
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/mondo/mondo.obo
  /content/drive/MyDrive/Colab Notebooks/dataverse_shepherd/preprocess/mondo/mondo_references.csv


In [ ]:
import pandas as pd, pickle, json, numpy as np

# Load disease category mapping
cat_df = pd.read_csv(f"{SD}/preprocess/orphanet/categorization_of_orphanet_diseases.csv")
print(f"Category mapping: {cat_df.shape}")
print(cat_df.columns.tolist())
print(cat_df.head())

# Load orphanet to mondo mapping
with open(f"{SD}/preprocess/orphanet/orphanet_to_mondo_dict.pkl", 'rb') as f:
    orpha_to_mondo = pickle.load(f)

# Load mondo to name
with open(f"{SD}/knowledge_graph/8.9.21_kg/mondo_to_name_dict_8.9.21_kg.pkl", 'rb') as f:
    mondo_to_name = pickle.load(f)
with open(f"{SD}/knowledge_graph/8.9.21_kg/mondo_to_idx_dict_8.9.21_kg.pkl", 'rb') as f:
    mondo_to_idx = pickle.load(f)

print(f"\nOrphanet to MONDO mappings: {len(orpha_to_mondo)}")
print(f"Sample: {list(orpha_to_mondo.items())[:3]}")

# Check category df
print(f"\nSample categories:")
print(cat_df.head(10))

Category mapping: (7225, 4)
['Unnamed: 0', 'OrphaNumber', 'Disorder_Name', 'Category']
   Unnamed: 0  OrphaNumber                                      Disorder_Name  \
0           0       166024      Multiple epiphyseal dysplasia, Al-Gazali type   
1           1           58                                  Alexander disease   
2           2       166032  Multiple epiphyseal dysplasia, with miniepiphyses   
3           3           61                                 Alpha-mannosidosis   
4           4       166029  Multiple epiphyseal dysplasia, with severe pro...   

                           Category  
0                 Rare bone disease  
1           Rare neurologic disease  
2                 Rare bone disease  
3  Rare inborn errors of metabolism  
4                 Rare bone disease  

Orphanet to MONDO mappings: 9576
Sample: [(48652, ['11652']), (2491, ['7795']), (435934, ['18568'])]

Sample categories:
   Unnamed: 0  OrphaNumber                                      Disorder_Nam

In [ ]:
# Build MONDO -> category mapping via orphanet
mondo_to_category = {}
for orpha_num, mondo_ids in orpha_to_mondo.items():
    cat_row = cat_df[cat_df['OrphaNumber'] == orpha_num]
    if len(cat_row) > 0:
        category = cat_row.iloc[0]['Category']
        for mid in mondo_ids:
            mondo_to_category[str(mid)] = category

# Build disease name -> category mapping for scoring
name_to_category = {}
for mondo_id, name in mondo_to_name.items():
    if str(mondo_id) in mondo_to_category:
        name_to_category[name] = mondo_to_category[str(mondo_id)]

print(f"Diseases with category: {len(name_to_category)} / {len(mondo_to_name)}")
print(f"Unique categories: {len(set(name_to_category.values()))}")

# Load scores
scores_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/split/shepherd_disease_char_scores_v3.csv')
patients_v3 = [json.loads(l) for l in open("/content/drive/MyDrive/Colab Notebooks/split/shepherd_test_patients_v3.jsonl")]

# For each patient: get true disease category, check top-10 ranked diseases' categories
results = []
for p in patients_v3:
    pid = p['id']
    true_mondos = [str(d) for d in p['true_diseases']]
    true_categories = set(mondo_to_category.get(d, 'Unknown') for d in true_mondos)
    true_categories.discard('Unknown')

    if not true_categories:
        continue

    pdf = scores_df[scores_df['patient_id'] == pid].sort_values('similarities', ascending=False).reset_index(drop=True)

    # Top-k diseases and their categories
    for k in [1, 5, 10, 20]:
        topk = pdf.head(k)
        topk_categories = [name_to_category.get(d, 'Unknown') for d in topk['diseases']]
        topk_categories = [c for c in topk_categories if c != 'Unknown']

        if topk_categories:
            match_rate = sum(1 for c in topk_categories if c in true_categories) / len(topk_categories)
        else:
            match_rate = 0

        results.append({'pid': pid, 'k': k, 'match_rate': match_rate, 'true_categories': list(true_categories)})

results_df = pd.DataFrame(results)

print(f"\n--- SHEPHERD Disease Characterization (Category-level) ---")
print(f"Patients evaluated: {results_df['pid'].nunique()}")
for k in [1, 5, 10, 20]:
    subset = results_df[results_df['k'] == k]
    mean_match = subset['match_rate'].mean()
    print(f"Top-{k}: {mean_match:.3f} category match rate (random ≈ {1/33:.3f} = 0.030)")

# Also: what fraction of patients have correct category in top-1?
top1 = results_df[results_df['k'] == 1]
print(f"\nTop-1 exact category match: {(top1['match_rate'] > 0).mean():.3f}")

Diseases with category: 6714 / 21233
Unique categories: 28

--- SHEPHERD Disease Characterization (Category-level) ---
Patients evaluated: 57
Top-1: 0.000 category match rate (random ≈ 0.030 = 0.030)
Top-5: 0.018 category match rate (random ≈ 0.030 = 0.030)
Top-10: 0.044 category match rate (random ≈ 0.030 = 0.030)
Top-20: 0.037 category match rate (random ≈ 0.030 = 0.030)

Top-1 exact category match: 0.000


In [ ]:
# Check their environment.yml for exact versions
!cat /content/SHEPHERD/environment.yml

name: shepherd
channels:
  - metric-learning
  - biobuilds
  - anaconda
  - pytorch
  - conda-forge
  - defaults
dependencies:
  - _libgcc_mutex=0.1
  - _openmp_mutex=4.5
  - absl-py=0.12.0
  - aiohttp=3.7.4
  - argon2-cffi=20.1.0
  - async-timeout=3.0.1
  - async_generator=1.10
  - attrs=20.3.0
  - backcall=0.2.0
  - blas=1.0
  - bleach=3.3.0
  - blinker=1.4
  - bokeh=2.3.1
  - brotlipy=0.7.0
  - bzip2=1.0.8
  - c-ares=1.17.1
  - ca-certificates=2021.5.30
  - cachetools=4.2.1
  - certifi=2021.5.30
  - cffi=1.14.5
  - click=7.1.2
  - cloudpickle=1.6.0
  - colorcet=2.0.6
  - cryptography=3.4.7
  - cudatoolkit=10.2.89
  - cytoolz=0.11.0
  - dask=2021.4.0
  - dask-core=2021.4.0
  - datashader=0.11.1
  - datashape=0.5.4
  - dbus=1.13.18
  - defusedxml=0.7.1
  - distributed=2021.4.0
  - entrypoints=0.3
  - expat=2.3.0
  - faiss-gpu=1.7.0
  - ffmpeg=4.3
  - fontconfig=2.13.1
  - freetype=2.10.4
  - future=0.18.2
  - glib=2.68.0
  - gmp=6.2.1
  - gnutls=3.6.5
  - google-auth=1.28.0
  - google

In [ ]:
!cat /content/SHEPHERD/install_pyg.sh

pip install torch-scatter==2.0.8 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
pip install torch-sparse==0.6.11 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
pip install torch-cluster==1.5.9 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
pip install torch-spline-conv==1.2.1 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
pip install torch-geometric==1.7.2

In [ ]:
# Install conda in Colab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


In [ ]:
import condacolab
condacolab.check()

# Create SHEPHERD's exact environment
# We can't use their full environment.yml (too many deps),
# but we can install just the critical packages
!conda create -n shepherd python=3.8 -y
!conda run -n shepherd pip install torch==1.8.0+cu102 torchvision==0.9.0+cu102 torchaudio==0.8.0 -f https://download.pytorch.org/whl/torch_stable.html
!conda run -n shepherd pip install torch-scatter==2.0.8 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
!conda run -n shepherd pip install torch-sparse==0.6.11 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
!conda run -n shepherd pip install torch-cluster==1.5.9 -f https://pytorch-geometric.com/whl/torch-1.8.0+cu102.html
!conda run -n shepherd pip install torch-geometric==1.7.2
!conda run -n shepherd pip install pytorch-lightning==1.4.5 wandb jsonlines obonet allennlp==2.4.0 pytorch-metric-learning==0.9.98
!conda run -n shepherd python -c "import torch; import torch_geometric; print(f'PyTorch: {torch.__version__}, PyG: {torch_geometric.__version__}')"

✨🍰✨ Everything looks OK!
Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/shepherd

  added / updated specs:
    - python=3.8


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    icu-78.3                   |       h33c6efd_0        12.1 MB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_101         709 KB  conda-forge
    libffi-3.5.2               |       h3435931_0     